# Proves dels repositoris F1

Aquest notebook comprova els mètodes CRUD dels repositoris, consultes específiques, paginació i l'operació concreta per afegir pneumàtics a un resultat.


In [ ]:
import os
import sys
from pathlib import Path

arrel = Path.cwd()
if arrel.name == "notebooks":
    arrel = arrel.parent

sys.path.insert(0, str(arrel / "src"))

bbdd = arrel / "notebooks" / "f1_repositoris.sqlite"
if bbdd.exists():
    bbdd.unlink()

os.environ["ENVIRONMENT"] = "test"
os.environ["DB_URL_TEST"] = f"sqlite:///{bbdd.as_posix()}"

from datetime import date

from domain.db import Base, engine
from domain.models import Conductor, Cotxe, Cursa, Pista, Pneumatic, ResultatCursa
from domain.repositories import UnitatDeTreball

Base.metadata.create_all(engine)
print("Base de dades de test creada:", bbdd)


## 1. Creació inicial de dades


In [ ]:
with UnitatDeTreball() as uow:
    verstappen = uow.conductors.add(
        Conductor(nom="Max Verstappen", nacionalitat="Països Baixos", numero=1)
    )
    alonso = uow.conductors.add(
        Conductor(nom="Fernando Alonso", nacionalitat="Espanya", numero=14)
    )
    norris = uow.conductors.add(
        Conductor(nom="Lando Norris", nacionalitat="Regne Unit", numero=4)
    )

    red_bull = uow.cotxes.add(
        Cotxe(
            codi_xassis="RB20-01",
            proveidor_de_motors="Honda RBPT",
            potencia_cv=1050,
            pes_kg=798.0,
            id_driver=verstappen.id_driver,
        )
    )

    montmelo = uow.pistes.add(
        Pista(nom="Circuit de Barcelona-Catalunya", pais="Espanya", longitud_km=4.657)
    )
    monaco = uow.pistes.add(
        Pista(nom="Circuit de Monaco", pais="Mònaco", longitud_km=3.337)
    )

    dur = uow.pneumatics.add(Pneumatic(nom_compost="Dur", descripcio="Compost amb més durabilitat"))
    mitja = uow.pneumatics.add(Pneumatic(nom_compost="Mitjà", descripcio="Compost equilibrat"))
    tou = uow.pneumatics.add(Pneumatic(nom_compost="Tou", descripcio="Compost ràpid però menys durable"))

    gp_espanya = uow.curses.add(
        Cursa(
            nom_del_gran_premi="Gran Premi d'Espanya",
            data=date(2025, 6, 1),
            temporada=2025,
            id_track=montmelo.id_track,
        )
    )
    gp_monaco = uow.curses.add(
        Cursa(
            nom_del_gran_premi="Gran Premi de Mònaco",
            data=date(2025, 5, 25),
            temporada=2025,
            id_track=monaco.id_track,
        )
    )

    resultat_1 = uow.resultats.add(
        ResultatCursa(
            posicio_final=1,
            punts=25,
            temps_total="1:31:44.742",
            id_driver=verstappen.id_driver,
            id_race=gp_espanya.id_race,
        )
    )
    resultat_2 = uow.resultats.add(
        ResultatCursa(
            posicio_final=2,
            punts=18,
            temps_total="+2.219",
            id_driver=alonso.id_driver,
            id_race=gp_espanya.id_race,
        )
    )
    resultat_3 = uow.resultats.add(
        ResultatCursa(
            posicio_final=3,
            punts=15,
            temps_total="+7.812",
            id_driver=norris.id_driver,
            id_race=gp_espanya.id_race,
        )
    )
    resultat_4 = uow.resultats.add(
        ResultatCursa(
            posicio_final=1,
            punts=25,
            temps_total="1:40:12.001",
            id_driver=verstappen.id_driver,
            id_race=gp_monaco.id_race,
        )
    )

    uow.commit()

ids = {
    "verstappen": verstappen.id_driver,
    "alonso": alonso.id_driver,
    "norris": norris.id_driver,
    "red_bull": red_bull.id_cotxe,
    "montmelo": montmelo.id_track,
    "monaco": monaco.id_track,
    "dur": dur.id_tyre,
    "mitja": mitja.id_tyre,
    "tou": tou.id_tyre,
    "gp_espanya": gp_espanya.id_race,
    "gp_monaco": gp_monaco.id_race,
    "resultat_1": resultat_1.id_result,
    "resultat_2": resultat_2.id_result,
    "resultat_3": resultat_3.id_result,
    "resultat_4": resultat_4.id_result,
}
ids


## 2. Repositori de conductors: add, get, get_all, update, delete i consultes específiques


In [ ]:
with UnitatDeTreball() as uow:
    conductor = uow.conductors.get(ids["verstappen"])
    assert conductor is not None
    assert conductor.nom == "Max Verstappen"

    tots = uow.conductors.get_all()
    assert len(tots) >= 3

    per_numero = uow.conductors.get_by_numero(14)
    assert per_numero is not None
    assert per_numero.nom == "Fernando Alonso"

    espanyols = uow.conductors.get_by_nacionalitat("Espanya")
    assert len(espanyols) == 1

    cerca_nom = uow.conductors.get_by_nom("Max")
    assert len(cerca_nom) == 1

    temporal = uow.conductors.add(Conductor(nom="Conductor Temporal", nacionalitat="Itàlia", numero=99))
    id_temporal = temporal.id_driver
    temporal.nom = "Conductor Actualitzat"
    uow.conductors.update(temporal)
    assert uow.conductors.get(id_temporal).nom == "Conductor Actualitzat"

    uow.conductors.delete(id_temporal)
    assert uow.conductors.get(id_temporal) is None
    uow.commit()

print("RepositoriConductor OK")


## 3. Repositori de cotxes


In [ ]:
with UnitatDeTreball() as uow:
    cotxe = uow.cotxes.get(ids["red_bull"])
    assert cotxe is not None
    assert cotxe.codi_xassis == "RB20-01"

    assert len(uow.cotxes.get_all()) >= 1
    assert uow.cotxes.get_by_codi_xassis("RB20-01").id_cotxe == ids["red_bull"]
    assert len(uow.cotxes.get_by_proveidor_de_motors("Honda RBPT")) == 1

    cotxe.potencia_cv = 1060
    uow.cotxes.update(cotxe)
    assert uow.cotxes.get(ids["red_bull"]).potencia_cv == 1060

    conductor_tmp = uow.conductors.add(Conductor(nom="Pilot Cotxe", nacionalitat="França", numero=31))
    cotxe_tmp = uow.cotxes.add(
        Cotxe(
            codi_xassis="TMP-01",
            proveidor_de_motors="Renault",
            potencia_cv=1000,
            pes_kg=798,
            id_driver=conductor_tmp.id_driver,
        )
    )
    id_cotxe_tmp = cotxe_tmp.id_cotxe
    uow.cotxes.delete(id_cotxe_tmp)
    assert uow.cotxes.get(id_cotxe_tmp) is None
    uow.commit()

print("RepositoriCotxe OK")


## 4. Repositori de pistes


In [ ]:
with UnitatDeTreball() as uow:
    pista = uow.pistes.get(ids["montmelo"])
    assert pista is not None
    assert pista.pais == "Espanya"

    assert len(uow.pistes.get_all()) >= 2
    assert len(uow.pistes.get_by_pais("Espanya")) == 1
    assert len(uow.pistes.get_by_nom("Barcelona")) == 1

    pista.longitud_km = 4.660
    uow.pistes.update(pista)
    assert round(uow.pistes.get(ids["montmelo"]).longitud_km, 3) == 4.660

    pista_tmp = uow.pistes.add(Pista(nom="Pista Temporal", pais="Portugal", longitud_km=4.6))
    id_pista_tmp = pista_tmp.id_track
    uow.pistes.delete(id_pista_tmp)
    assert uow.pistes.get(id_pista_tmp) is None
    uow.commit()

print("RepositoriPista OK")


## 5. Repositori de curses


In [ ]:
with UnitatDeTreball() as uow:
    cursa = uow.curses.get(ids["gp_espanya"])
    assert cursa is not None
    assert cursa.temporada == 2025

    assert len(uow.curses.get_all()) >= 2
    assert len(uow.curses.get_by_temporada(2025)) >= 2
    assert len(uow.curses.get_by_gran_premi("Espanya")) == 1

    cursa.nom_del_gran_premi = "Gran Premi d'Espanya 2025"
    uow.curses.update(cursa)
    assert "2025" in uow.curses.get(ids["gp_espanya"]).nom_del_gran_premi

    pista_tmp = uow.pistes.add(Pista(nom="Pista Cursa Temporal", pais="Alemanya", longitud_km=5.1))
    cursa_tmp = uow.curses.add(
        Cursa(
            nom_del_gran_premi="Gran Premi Temporal",
            data=date(2025, 7, 1),
            temporada=2025,
            id_track=pista_tmp.id_track,
        )
    )
    id_cursa_tmp = cursa_tmp.id_race
    uow.curses.delete(id_cursa_tmp)
    assert uow.curses.get(id_cursa_tmp) is None
    uow.commit()

print("RepositoriCursa OK")


## 6. Repositori de pneumàtics


In [ ]:
with UnitatDeTreball() as uow:
    pneumatic = uow.pneumatics.get(ids["dur"])
    assert pneumatic is not None
    assert pneumatic.nom_compost == "Dur"

    assert len(uow.pneumatics.get_all()) >= 3
    assert uow.pneumatics.get_by_nom_compost("Mitjà").id_tyre == ids["mitja"]

    pneumatic.descripcio = "Compost pensat per tandes llargues"
    uow.pneumatics.update(pneumatic)
    assert "llargues" in uow.pneumatics.get(ids["dur"]).descripcio

    pneumatic_tmp = uow.pneumatics.add(Pneumatic(nom_compost="Experimental", descripcio="Només test"))
    id_pneumatic_tmp = pneumatic_tmp.id_tyre
    uow.pneumatics.delete(id_pneumatic_tmp)
    assert uow.pneumatics.get(id_pneumatic_tmp) is None
    uow.commit()

print("RepositoriPneumatic OK")


## 7. Repositori de resultats: CRUD, filtres, podi, paginació i operació de domini


In [ ]:
with UnitatDeTreball() as uow:
    resultat = uow.resultats.get(ids["resultat_1"])
    assert resultat is not None
    assert resultat.posicio_final == 1

    assert len(uow.resultats.get_all()) >= 4
    assert len(uow.resultats.get_by_conductor(ids["verstappen"])) == 2
    assert len(uow.resultats.get_by_cursa(ids["gp_espanya"])) == 3

    podi = uow.resultats.get_podi(ids["gp_espanya"])
    assert [r.posicio_final for r in podi] == [1, 2, 3]

    pagina_1 = uow.resultats.get_paginated(page=1, page_size=2)
    pagina_2 = uow.resultats.get_paginated(page=2, page_size=2)
    assert len(pagina_1) == 2
    assert len(pagina_2) >= 2

    resultat.punts = 26
    uow.resultats.update(resultat)
    assert uow.resultats.get(ids["resultat_1"]).punts == 26

    us = uow.resultats.afegir_pneumatic_a_resultat(
        id_result=ids["resultat_1"],
        id_tyre=ids["mitja"],
        periode=1,
        nombre_de_voltes=31,
        numero_us=1,
    )
    assert us.id_result == ids["resultat_1"]
    assert us.id_tyre == ids["mitja"]

    conductor_tmp = uow.conductors.add(Conductor(nom="Pilot Resultat", nacionalitat="Canadà", numero=18))
    cursa_tmp = uow.curses.add(
        Cursa(
            nom_del_gran_premi="Gran Premi Resultat Temporal",
            data=date(2025, 8, 1),
            temporada=2025,
            id_track=ids["montmelo"],
        )
    )
    resultat_tmp = uow.resultats.add(
        ResultatCursa(
            posicio_final=10,
            punts=1,
            temps_total="+1 volta",
            id_driver=conductor_tmp.id_driver,
            id_race=cursa_tmp.id_race,
        )
    )
    id_resultat_tmp = resultat_tmp.id_result
    uow.resultats.delete(id_resultat_tmp)
    assert uow.resultats.get(id_resultat_tmp) is None

    uow.commit()

print("RepositoriResultatCursa OK")


## 8. Unit of Work, rollback i camp last_update

Aquesta prova comprova que `rollback()` desfà els canvis i que `last_update` canvia quan una entitat es modifica.


In [ ]:
import time

with UnitatDeTreball() as uow:
    conductor = uow.conductors.get(ids["verstappen"])
    assert conductor is not None
    last_update_abans = conductor.last_update

time.sleep(1.1)

with UnitatDeTreball() as uow:
    conductor = uow.conductors.get(ids["verstappen"])
    conductor.nom = "Max Verstappen Actualitzat"
    uow.conductors.update(conductor)
    uow.commit()

with UnitatDeTreball() as uow:
    conductor = uow.conductors.get(ids["verstappen"])
    assert conductor.nom == "Max Verstappen Actualitzat"
    assert conductor.last_update is not None
    assert conductor.last_update >= last_update_abans
    conductor.nom = "Max Verstappen"
    uow.conductors.update(conductor)
    uow.commit()

with UnitatDeTreball() as uow:
    temporal = uow.conductors.add(Conductor(nom="Rollback Test", nacionalitat="Test", numero=77))
    id_temporal = temporal.id_driver
    uow.rollback()

with UnitatDeTreball() as uow:
    assert uow.conductors.get(id_temporal) is None

print("UnitOfWork, rollback i last_update OK")


## 9. Comprovació final


In [ ]:
with UnitatDeTreball() as uow:
    assert len(uow.conductors.get_all()) >= 4
    assert len(uow.curses.get_all()) >= 3
    assert len(uow.resultats.get_all()) >= 4

print("Totes les proves han acabat sense errors.")
